In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/capstone_deepfake/capstone_deepfake'

# zip 파일명 실제 확인
print('ROOT 직속 zip/tar:')
for x in sorted(os.listdir(PROJECT_ROOT)):
    if x.endswith(('.zip', '.tar', '.tar.gz')):
        full = os.path.join(PROJECT_ROOT, x)
        print(f'  {x}  ({os.path.getsize(full)//1024//1024}MB)')

# 통합 CSV도 있는지
print('\ncsv_dataset_v4:', os.listdir(f'{PROJECT_ROOT}/csv_dataset_v4') if os.path.isdir(f'{PROJECT_ROOT}/csv_dataset_v4') else '없음')

Mounted at /content/drive
ROOT 직속 zip/tar:
  kodf_crops.tar.gz  (84MB)
  test.zip  (135MB)
  train.zip  (583MB)
  val.zip  (127MB)

csv_dataset_v4: ['train_v4.csv', 'val_v4.csv', 'test_v4.csv']


In [ ]:
import zipfile, tarfile, time, os, glob

LOCAL = '/content/data'
os.makedirs(LOCAL, exist_ok=True)

# dataset_B zip 3개
for s in ['train', 'val', 'test']:
    src = f'{PROJECT_ROOT}/{s}.zip'
    if not os.path.exists(src):
        print(f'!! 없음: {src} — 실제 zip 파일명 확인 필요')
        continue
    t0 = time.time()
    with zipfile.ZipFile(src) as zf:
        zf.extractall(LOCAL)
    print(f'{s}.zip 해제 {round(time.time()-t0)}초')

# kodf tar
t0 = time.time()
with tarfile.open(f'{PROJECT_ROOT}/kodf_crops.tar.gz') as tf:
    tf.extractall(LOCAL)
print(f'kodf 해제 {round(time.time()-t0)}초')

# 해제 후 구조 확인 — 매핑 경로 결정용
print('\nLOCAL 직속:', sorted(os.listdir(LOCAL)))
for root, dirs, files in os.walk(LOCAL):
    depth = root.replace(LOCAL, '').count('/')
    if depth <= 2 and (dirs or files):
        jpgs = [f for f in files if f.endswith('.jpg')]
        print(f'{root.replace(LOCAL,"LOCAL")}/  (하위폴더 {len(dirs)}, jpg {len(jpgs)})')
        if depth >= 2:  # 너무 깊이 안 들어감
            dirs[:] = []

train.zip 해제 16초
val.zip 해제 6초
test.zip 해제 6초


/tmp/ipykernel_1062/927808987.py:20: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(LOCAL)


kodf 해제 4초

LOCAL 직속: ['done_audio_driven.txt', 'done_dffs.txt', 'done_dfl.txt', 'done_fo.txt', 'done_fsgan.txt', 'done_real.txt', 'kodf_audio_driven_crops.csv', 'kodf_crops', 'kodf_dffs_crops.csv', 'kodf_dfl_crops.csv', 'kodf_fo_crops.csv', 'kodf_fsgan_crops.csv', 'kodf_real_crops.csv', 'test', 'train', 'val']
LOCAL/  (하위폴더 4, jpg 0)
LOCAL/kodf_crops/  (하위폴더 6, jpg 0)
LOCAL/kodf_crops/dffs/  (하위폴더 21, jpg 0)
LOCAL/kodf_crops/audio_driven/  (하위폴더 10, jpg 0)
LOCAL/kodf_crops/dfl/  (하위폴더 9, jpg 0)
LOCAL/kodf_crops/fo/  (하위폴더 25, jpg 0)
LOCAL/kodf_crops/real/  (하위폴더 20, jpg 0)
LOCAL/kodf_crops/fsgan/  (하위폴더 11, jpg 0)
LOCAL/test/  (하위폴더 0, jpg 11314)
LOCAL/train/  (하위폴더 0, jpg 48167)
LOCAL/val/  (하위폴더 0, jpg 10760)


모델학습


In [ ]:
!pip install -q timm

import os, time, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm

# 재현성
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = '/content/drive/MyDrive/capstone_deepfake/capstone_deepfake'
LOCAL = '/content/data'
CSV_DIR = f'{PROJECT_ROOT}/csv_dataset_v4'
CKPT_DIR = f'{PROJECT_ROOT}/ckpt_v4'      # Drive에 저장 (resume용)
os.makedirs(CKPT_DIR, exist_ok=True)

# 하이퍼파라미터 (확정)
EPOCHS = 10
LR = 5e-5
BATCH = 64
WD = 0.05
PATIENCE = 3
NUM_WORKERS = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

device: cuda | GPU: Tesla T4


In [ ]:
def map_local(fp):
    if '/dataset_B_recrop/' in fp:
        return LOCAL + '/' + fp.split('/dataset_B_recrop/')[1]
    if '/kodf_crops/' in fp:
        return LOCAL + '/kodf_crops/' + fp.split('/kodf_crops/')[1]
    return fp

EVAL_TF = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.485,.456,.406], std=[.229,.224,.225]),
])
TRAIN_TF = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.485,.456,.406], std=[.229,.224,.225]),
])

class DFDataset(Dataset):
    def __init__(self, csv_path, tf):
        df = pd.read_csv(csv_path)
        df['local'] = df['filepath'].map(map_local)
        before = len(df)
        df = df[df['local'].map(os.path.exists)].reset_index(drop=True)  # 누락 파일 자동 제외
        dropped = before - len(df)
        if dropped: print(f'  [{os.path.basename(csv_path)}] 누락 {dropped}개 제외 → {len(df)}행')
        self.paths = df['local'].tolist()
        self.labels = df['label'].astype(int).tolist()
        self.tf = tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        return self.tf(img), self.labels[i]

train_ds = DFDataset(f'{CSV_DIR}/train_v4.csv', TRAIN_TF)
val_ds   = DFDataset(f'{CSV_DIR}/val_v4.csv',   EVAL_TF)
print(f'train {len(train_ds)} / val {len(val_ds)}')

train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_dl   = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
from sklearn.metrics import f1_score

model = timm.create_model('convnextv2_tiny.fcmae_ft_in22k_in1k',
                          pretrained=True, num_classes=2).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
crit = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler()

LAST = f'{CKPT_DIR}/last_checkpoint.pt'
BEST = f'{CKPT_DIR}/best_model.pt'

# ---- resume ----
start_epoch = 0
best_f1 = 0.0
no_improve = 0
if os.path.exists(LAST):
    ck = torch.load(LAST, map_location=device)
    model.load_state_dict(ck['model'])
    opt.load_state_dict(ck['opt'])
    sched.load_state_dict(ck['sched'])
    scaler.load_state_dict(ck['scaler'])
    start_epoch = ck['epoch'] + 1
    best_f1 = ck['best_f1']
    no_improve = ck['no_improve']
    print(f'resume: epoch {start_epoch}부터, best_f1={best_f1:.4f}')
else:
    print('새 학습 시작')

@torch.no_grad()
def evaluate():
    model.eval()
    preds, gts = [], []
    for x, y in val_dl:
        x = x.to(device, non_blocking=True)
        with torch.cuda.amp.autocast():
            logit = model(x)
        preds += logit.argmax(1).cpu().tolist()
        gts += y.tolist()
    # label 0=Fake, 1=Real. F1은 양쪽 macro로 (in-domain 비교용)
    return f1_score(gts, preds, average='macro'), f1_score(gts, preds, pos_label=0)

# ---- 학습 ----
for epoch in range(start_epoch, EPOCHS):
    model.train()
    t0 = time.time()
    run_loss = 0.0
    for i, (x, y) in enumerate(train_dl):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad()
        with torch.cuda.amp.autocast():
            loss = crit(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        run_loss += loss.item()
        if i % 100 == 0:
            print(f'  e{epoch} [{i}/{len(train_dl)}] loss {loss.item():.4f}')
    sched.step()

    macro_f1, fake_f1 = evaluate()
    dt = round(time.time()-t0)
    print(f'[epoch {epoch}] loss {run_loss/len(train_dl):.4f} | val macroF1 {macro_f1:.4f} fakeF1 {fake_f1:.4f} | {dt}s')

    # best 갱신
    improved = macro_f1 > best_f1
    if improved:
        best_f1 = macro_f1; no_improve = 0
        torch.save({'model': model.state_dict(), 'epoch': epoch, 'macro_f1': macro_f1}, BEST)
        print(f'  ✓ best 갱신 {best_f1:.4f}')
    else:
        no_improve += 1

    # last 체크포인트 (Drive, resume용) — 매 epoch
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                'sched': sched.state_dict(), 'scaler': scaler.state_dict(),
                'epoch': epoch, 'best_f1': best_f1, 'no_improve': no_improve}, LAST)

    if no_improve >= PATIENCE:
        print(f'early stop (no improve {no_improve})'); break

print('학습 종료. best macroF1:', best_f1)

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

새 학습 시작


/tmp/ipykernel_1062/2529946832.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_1062/2529946832.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  e0 [0/782] loss 0.7607
  e0 [100/782] loss 0.4846
  e0 [200/782] loss 0.3442
  e0 [300/782] loss 0.3273
  e0 [400/782] loss 0.2711
  e0 [500/782] loss 0.2223
  e0 [600/782] loss 0.2529
  e0 [700/782] loss 0.2978


/tmp/ipykernel_1062/2529946832.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


...

[epoch 8] loss 0.1992 | val macroF1 0.9465 fakeF1 0.9469 | 508s
  ✓ best 갱신 0.9465


/tmp/ipykernel_1062/2529946832.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  e9 [0/782] loss 0.1987
  e9 [100/782] loss 0.1987
  e9 [200/782] loss 0.1986
  e9 [300/782] loss 0.1987
  e9 [400/782] loss 0.1986
  e9 [500/782] loss 0.2007
  e9 [600/782] loss 0.1987
  e9 [700/782] loss 0.1986


/tmp/ipykernel_1062/2529946832.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[epoch 9] loss 0.1990 | val macroF1 0.9467 fakeF1 0.9469 | 508s
  ✓ best 갱신 0.9467
학습 종료. best macroF1: 0.94672478087271


 v4 test 평가


In [ ]:
import torch, pandas as pd, os
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, classification_report

# best 로드
ck = torch.load(f'{CKPT_DIR}/best_model.pt', map_location=device)
model.load_state_dict(ck['model'])
model.eval()
print('best 로드: epoch', ck['epoch'], 'macroF1', round(ck['macro_f1'],4))

# test set (메타 보존 위해 df도 들고감)
test_df = pd.read_csv(f'{CSV_DIR}/test_v4.csv')
test_df['local'] = test_df['filepath'].map(map_local)
test_df = test_df[test_df['local'].map(os.path.exists)].reset_index(drop=True)
test_ds = DFDataset(f'{CSV_DIR}/test_v4.csv', EVAL_TF)
test_dl = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# 추론
preds, probs = [], []
with torch.no_grad():
    for x, _ in test_dl:
        x = x.to(device, non_blocking=True)
        with torch.cuda.amp.autocast():
            logit = model(x)
        p = torch.softmax(logit.float(), 1)
        preds += logit.argmax(1).cpu().tolist()
        probs += p[:,1].cpu().tolist()   # P(Real)

test_df = test_df.iloc[:len(preds)].copy()
test_df['pred'] = preds
test_df['p_real'] = probs

# 전체
y, yhat = test_df['label'].tolist(), test_df['pred'].tolist()
print('\n=== 전체 test ===')
print('macroF1:', round(f1_score(y,yhat,average='macro'),4),
      '| fakeF1:', round(f1_score(y,yhat,pos_label=0),4),
      '| n:', len(test_df))

# source별
print('\n=== source별 ===')
for src, g in test_df.groupby('source'):
    mf = f1_score(g.label, g.pred, average='macro')
    ff = f1_score(g.label, g.pred, pos_label=0) if (g.label==0).any() and (g.pred==0).any() else float('nan')
    print(f'{src:8s} n={len(g):5d}  macroF1 {mf:.4f}  fakeF1 {ff:.4f}')

# fake_type별 (fake만, recall = 잡아낸 비율)
print('\n=== fake_type별 (fake recall) ===')
fakes = test_df[test_df.label==0]
for ft, g in fakes.groupby('fake_type'):
    recall = (g.pred==0).mean()
    print(f'{ft:15s} n={len(g):5d}  recall {recall:.4f}')

# 저장 (보고서/추가분석용)
test_df.to_csv(f'{PROJECT_ROOT}/v4_test_predictions.csv', index=False)
print('\n예측 저장 → v4_test_predictions.csv')

best 로드: epoch 9 macroF1 0.9467


/tmp/ipykernel_1062/3071250155.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



=== 전체 test ===
macroF1: 0.943 | fakeF1: 0.9432 | n: 11914

=== source별 ===
celebdf  n= 6514  macroF1 0.9693  fakeF1 0.9694
ffpp     n= 4800  macroF1 0.9145  fakeF1 0.9099
kodf     n=  600  macroF1 0.8478  fakeF1 0.9213

=== fake_type별 (fake recall) ===
CelebDF_Swap    n= 3260  recall 0.9702
Deepfakes       n=  580  recall 0.9431
Face2Face       n=  320  recall 0.9625
FaceShifter     n=  400  recall 0.8825
FaceSwap        n=  600  recall 0.9267
NeuralTextures  n=  500  recall 0.6040
audio_driven    n=   60  recall 0.0000
dffs            n=  120  recall 1.0000
dfl             n=   60  recall 1.0000
fo              n=  120  recall 0.9250
fsgan           n=  120  recall 0.9917

예측 저장 → v4_test_predictions.csv


In [ ]:
import pandas as pd, numpy as np

pred = pd.read_csv(f'{PROJECT_ROOT}/v4_test_predictions.csv')

# label 0=Fake, 1=Real. P(Real) 높을수록 Real로 판단.
# fake(label=0)인데 p_real 높으면 → real로 오판
def show(ft):
    g = pred[(pred.label==0) & (pred.fake_type==ft)]
    pr = g['p_real'].values
    print(f'\n=== {ft} (fake {len(g)}장) ===')
    print(f'  p_real  min {pr.min():.3f} / median {np.median(pr):.3f} / max {pr.max():.3f}')
    print(f'  잡음(p_real<0.5, fake로 맞춤): {(pr<0.5).sum()}장')
    print(f'  놓침(p_real>=0.5, real로 오판): {(pr>=0.5).sum()}장')
    # 분포 구간
    bins = [0,0.1,0.3,0.5,0.7,0.9,1.01]
    hist, _ = np.histogram(pr, bins=bins)
    labels = ['0.0-0.1','0.1-0.3','0.3-0.5','0.5-0.7','0.7-0.9','0.9-1.0']
    for l,h in zip(labels,hist):
        print(f'    {l}: {"█"*h} {h}')

for ft in ['audio_driven', 'NeuralTextures', 'fo', 'dffs']:  # fo/dffs는 잘 잡는 KoDF 기법 — 대조군
    show(ft)

# KoDF real도 — real을 fake로 오판하는지 (macroF1<fakeF1 원인)
g = pred[(pred.label==1) & (pred.source=='kodf')]
print(f'\n=== KoDF real ({len(g)}장) ===')
pr = g['p_real'].values
print(f'  p_real median {np.median(pr):.3f}')
print(f'  real로 맞춤(p_real>=0.5): {(pr>=0.5).sum()}장 / fake로 오판: {(pr<0.5).sum()}장')


=== audio_driven (fake 60장) ===
  p_real  min 0.610 / median 0.933 / max 0.948
  잡음(p_real<0.5, fake로 맞춤): 0장
  놓침(p_real>=0.5, real로 오판): 60장
    0.0-0.1:  0
    0.1-0.3:  0
    0.3-0.5:  0
    0.5-0.7: █ 1
    0.7-0.9: ████████████ 12
    0.9-1.0: ███████████████████████████████████████████████ 47

=== NeuralTextures (fake 500장) ===
  p_real  min 0.033 / median 0.148 / max 0.959
  잡음(p_real<0.5, fake로 맞춤): 302장
  놓침(p_real>=0.5, real로 오판): 198장
    0.0-0.1: ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████ 227
    0.1-0.3: █████████████████████████████████████████████████████████ 57
    0.3-0.5: ██████████████████ 18
    0.5-0.7: ██████████████████ 18
    0.7-0.9: ████████████████████████████████████████████ 44
    0.9-1.0: █████████████████████████████████████████████████████████████████████████████████████